<a href="https://colab.research.google.com/github/vikadubanevica/Prompt-Engineering-with-Llama/blob/main/L3_multi_turn_conversations.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lesson 3
Import llama helper function:

In [3]:
from utils import llama

### LLMs are stateless
LLMs don't remember your previous interactions with them by default.

In [5]:
import requests
import json
import time

def llama(prompt, add_inst=True, model=None, temperature=0.7, max_tokens=256, verbose=False, url=None, headers=None, base=None, max_tries=3):
    # Default values setup
    if url is None:
        url = "http://localhost:11434/api/generate"
    if headers is None:
        headers = {'Content-Type': 'application/json'}
    if model is None:
        model = "llama2"

    full_prompt = f"<s>[INST] {prompt} [/INST]" if add_inst else prompt

    data = {
        "model": model,
        "prompt": full_prompt,
        "temperature": temperature,
        "max_tokens": max_tokens,
        "stream": False
    }

    response = None
    for attempt in range(max_tries):
        try:
            if verbose:
                print(f"Attempt {attempt + 1}/{max_tries}")
                print(f"Sending prompt (first 100 chars): {full_prompt[:100]}...")

            response = requests.post(url, headers=headers, json=data)

            if verbose:
                print(f"API Response Status Code: {response.status_code}")
                print(f"API Raw Response Content: {response.text}")

            response.raise_for_status() # Raises HTTPError for bad responses (4xx or 5xx)

            # Attempt to parse JSON and extract text
            json_response = response.json()
            if 'output' in json_response and 'choices' in json_response['output'] and len(json_response['output']['choices']) > 0:
                return json_response['output']['choices'][0]['text']
            elif 'response' in json_response: # Alternative common key
                return json_response['response']
            else:
                # If expected keys not found, return string representation of the whole JSON
                return json.dumps(json_response, indent=2) # Pretty print JSON

        except requests.exceptions.HTTPError as e:
            if verbose:
                print(f"HTTP Error during API call: {e}")
                if response is not None:
                    print(f"Status Code: {response.status_code}")
                    print(f"Raw Response: {response.text}")
            return f"Error: HTTP {response.status_code}. Raw Response: {response.text}"
        except json.JSONDecodeError as e:
            # This specifically catches the JSON parsing error
            if verbose:
                print(f"JSONDecodeError during parsing: {e}")
                if response is not None:
                    print(f"Status Code: {response.status_code}")
                    print(f"Raw Response that caused JSON error: {response.text}")
            return f"Error: JSONDecodeError. Raw Response: {response.text}"
        except requests.exceptions.ConnectionError as e:
            if verbose:
                print(f"Connection Error: {e}")
            print(f"Connection error. Retrying in 1 second...")
            time.sleep(1) # Wait before retrying
        except Exception as e:
            # Catch all other unexpected errors
            if verbose:
                print(f"An unexpected error occurred: {e}")
                if response is not None:
                    print(f"Status Code: {response.status_code}")
                    print(f"Raw Response: {response.text}")
            return f"An unexpected error occurred: {e}. Raw Response: {response.text if response else 'No response'}"

    return "Failed to get a response after multiple retries."

def llama_chat(prompts, responses, add_inst=False, model=None, temperature=0.7, max_tokens=256, verbose=False, url=None, headers=None, base=None, max_tries=3):
    if len(prompts) != len(responses) + 1:
        raise ValueError("Number of prompts must be one more than the number of responses for a chat.")

    chat_history = ""
    for i in range(len(responses)):
        chat_history += f"<s>[INST] {prompts[i]} [/INST]\n{responses[i]}\n</s>\n"

    current_prompt = prompts[-1]
    chat_history += f"<s>[INST] {current_prompt} [/INST]"

    return llama(chat_history, add_inst=add_inst, model=model, temperature=temperature, max_tokens=max_tokens, verbose=verbose, url=url, headers=headers, base=None, max_tries=max_tries)


In [14]:
prompt = """
    What are fun activities I can do this weekend?
"""
response = llama(prompt)
print(response)

Great question! There are so many fun activities you can do this weekend, depending on your interests and preferences. Here are some ideas to get you started:

1. Outdoor Adventures:
	* Go for a hike or a bike ride in a nearby park or nature reserve.
	* Take a kayaking or canoeing trip on a nearby lake or river.
	* Go camping or glamping (glamorous camping) in a nearby campsite.
	* Try rock climbing or bouldering at an indoor or outdoor climbing gym.
2. Cultural Events:
	* Attend a concert, play, or musical performance in your city or nearby.
	* Visit a museum or art gallery to see a new exhibit or collection.
	* Take a guided tour of a historic landmark or cultural site.
	* Attend a food festival or market to try new foods and drinks.
3. Sports and Fitness:
	* Join a recreational sports team or league, such as soccer, basketball, or volleyball.
	* Take a fitness class, such as yoga, Pilates, or spinning.
	* Go to a trampoline park or indoor climbing facility for a fun and active worko

In [15]:
prompt_2 = """
Which of these would be good for my health?
"""
response_2 = llama(prompt_2)
print(response_2)

I'm just an AI, I don't have access to personal health information or medical history, so I cannot provide personalized health advice. However, I can give you some general information about the two options you mentioned:

1. Exercise: Regular exercise is essential for maintaining good health. It can help prevent chronic diseases, such as heart disease, diabetes, and some types of cancer. Exercise can also improve mental health, boost mood, and increase energy levels.
2. Meditation: Meditation is a mindfulness practice that can have various physical and mental health benefits. It can help reduce stress, anxiety, and depression, improve sleep quality, and boost immune function. Some studies have also suggested that regular meditation practice may lead to changes in the brain that can improve cognitive function and emotional well-being.

Both exercise and meditation can be beneficial for overall health, but it's important to remember that they have different effects on the body and mind. 

### Constructing multi-turn prompts
You need to provide prior prompts and responses as part of the context of each new turn in the conversation.

In [16]:
prompt_1 = """
    What are fun activities I can do this weekend?
"""
response_1 = llama(prompt_1)

In [17]:
prompt_2 = """
Which of these would be good for my health?
"""

In [18]:
chat_prompt = f"""
<s>[INST] {prompt_1} [/INST]
{response_1}
</s>
<s>[INST] {prompt_2} [/INST]
"""
print(chat_prompt)


<s>[INST] 
    What are fun activities I can do this weekend?
 [/INST]
There are so many fun activities you can do this weekend, depending on your interests and preferences! Here are some ideas to get you started:

1. Outdoor Adventures:
	* Go for a hike or a bike ride in a nearby park or nature reserve.
	* Have a picnic or a barbecue in a scenic spot.
	* Try kayaking, paddleboarding, or rent a boat and explore a nearby lake or river.
2. Cultural Events:
	* Visit a museum or an art gallery to see a new exhibit or attend a special event.
	* Go to a concert or a play at a local theater.
	* Take a cooking class or a wine tasting to learn something new.
3. Sports and Fitness:
	* Join a recreational sports team or take a fitness class to get some exercise and meet new people.
	* Go to a trampoline park or an indoor climbing wall for a fun and active activity.
	* Try a new sport or activity, such as axe throwing or archery.
4. Family-Friendly Activities:
	* Take the kids to a children's mus

In [19]:
response_2 = llama(chat_prompt,
                 add_inst=False,
                 verbose=True)

Attempt 1/3
Sending prompt (first 100 chars): 
<s>[INST] 
    What are fun activities I can do this weekend?
 [/INST]
There are so many fun activi...
API Response Status Code: 200
API Raw Response Content: {"model":"llama2","created_at":"2026-03-08T20:04:22.445362607Z","response":"I'm glad you're interested in prioritizing your health! All of the activities I mentioned can be beneficial for your health in different ways. Here's a brief overview of each activity and how it can positively impact your health:\n\n1. Outdoor Adventures: Engaging in outdoor activities such as hiking, biking, or kayaking can improve your physical fitness, boost your mood, and reduce stress levels. Being in nature has also been shown to have a positive impact on mental health and overall well-being.\n2. Cultural Events: Attending cultural events such as concerts, plays, or art exhibits can enrich your cultural knowledge and appreciation, and provide an opportunity to socialize with like-minded individuals. The

In [20]:
print(response_2)

I'm glad you're interested in prioritizing your health! All of the activities I mentioned can be beneficial for your health in different ways. Here's a brief overview of each activity and how it can positively impact your health:

1. Outdoor Adventures: Engaging in outdoor activities such as hiking, biking, or kayaking can improve your physical fitness, boost your mood, and reduce stress levels. Being in nature has also been shown to have a positive impact on mental health and overall well-being.
2. Cultural Events: Attending cultural events such as concerts, plays, or art exhibits can enrich your cultural knowledge and appreciation, and provide an opportunity to socialize with like-minded individuals. These events can also help reduce stress and improve overall mental well-being.
3. Sports and Fitness: Engaging in sports or fitness activities can improve your physical fitness, boost your mood, and reduce stress levels. Exercise has also been shown to have numerous health benefits, inc

### Use llama chat helper function

In [21]:
prompt_1 = """
    What are fun activities I can do this weekend?
"""
response_1 = llama(prompt_1)

In [22]:
prompt_2 = """
Which of these would be good for my health?
"""

In [23]:
prompts = [prompt_1,prompt_2]
responses = [response_1]

In [24]:
# Pass prompts and responses to llama_chat function
response_2 = llama_chat(prompts,responses,verbose=True)

Attempt 1/3
Sending prompt (first 100 chars): <s>[INST] 
    What are fun activities I can do this weekend?
 [/INST]

There are plenty of fun acti...
API Response Status Code: 200
API Raw Response Content: {"model":"llama2","created_at":"2026-03-08T20:22:56.053469883Z","response":"I'm glad you're thinking about your health! All of the options I listed can be good for your health in different ways. Here are some more details about each option to help you decide which one might be best for you:\n\n1. Outdoor Adventures: Engaging in outdoor activities like hiking, camping, or kayaking can be great for your physical health by increasing your cardiovascular fitness, strengthening your muscles, and improving your flexibility. Being in nature can also have a positive impact on your mental health by reducing stress and improving your mood.\n2. Indoor Games: Playing indoor games like board games, card games, or video games can be a fun and social way to spend time with friends and family. These

In [25]:
print(response_2)

I'm glad you're thinking about your health! All of the options I listed can be good for your health in different ways. Here are some more details about each option to help you decide which one might be best for you:

1. Outdoor Adventures: Engaging in outdoor activities like hiking, camping, or kayaking can be great for your physical health by increasing your cardiovascular fitness, strengthening your muscles, and improving your flexibility. Being in nature can also have a positive impact on your mental health by reducing stress and improving your mood.
2. Indoor Games: Playing indoor games like board games, card games, or video games can be a fun and social way to spend time with friends and family. These activities can also help improve your cognitive function and problem-solving skills. However, it's important to make sure you're not spending too much time sitting and not getting enough physical activity.
3. Cultural Events: Attending cultural events like concerts, plays, or art exh

### Try it Yourself!

In [26]:
# replace prompt_3 with your own question!
prompt_3 = "Which of these activites would be fun with family?"
prompts = [prompt_1, prompt_2, prompt_3]
responses = [response_1, response_2]

response_3 = llama_chat(prompts, responses, verbose=True)

print(response_3)

Attempt 1/3
Sending prompt (first 100 chars): <s>[INST] 
    What are fun activities I can do this weekend?
 [/INST]

There are plenty of fun acti...
API Response Status Code: 200
API Raw Response Content: {"model":"llama2","created_at":"2026-03-08T20:33:03.602486515Z","response":"All of the activities I listed can be fun with family! However, the best activity for family bonding will depend on your family's interests and ages. Here are some suggestions based on common family interests:\n\n1. Outdoor Adventures: Hiking, camping, and kayaking are great activities for families who enjoy spending time together outdoors. You can explore nature, challenge yourselves physically, and spend quality time together.\n2. Indoor Games: Board games, card games, and video games are great ways to spend time together indoors. You can play games that are age-appropriate for your children and enjoy some friendly competition.\n3. Cultural Events: Attending concerts, plays, and art exhibits can be a great 

## Install Ollama

### Subtask:
Download and install the Ollama binary within the Colab environment. This will involve using `curl` and `sh` commands.


In [8]:
print('Installing zstd...')
!sudo apt-get update && sudo apt-get install -y zstd
print('zstd installed. Re-attempting Ollama installation...')
!curl -fsSL https://ollama.com/install.sh | sh
print('Ollama installation script executed again.')

Installing zstd...
Get:1 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:3 https://cli.github.com/packages stable InRelease [3,917 B]
Get:4 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Hit:5 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:6 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [85.2 kB]
Get:7 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:8 https://cli.github.com/packages stable/main amd64 Packages [357 B]
Get:9 http://security.ubuntu.com/ubuntu jammy-security/main amd64 Packages [3,780 kB]
Get:10 https://r2u.stat.illinois.edu/ubuntu jammy/main all Packages [9,824 kB]
Get:11 http://security.ubuntu.com/ubuntu jammy-security/universe amd64 Packages [1,301 kB]
Get:12 http://security.ubuntu.com/ubuntu jammy-security/multiverse amd64 Packages [62.6 kB]
Get:13 http://security.ubuntu.com/ubuntu jammy-security/

**Reasoning**:
Now that Ollama has been installed, the next step is to verify the installation and check its version to ensure it is correctly set up and executable.



In [9]:
print('Verifying Ollama installation...')
!ollama --version
print('Ollama version check complete.')

Verifying Ollama installation...
Ollama version check complete.


**Reasoning**:
Since Ollama is installed but not running, I need to start the Ollama server in the background to make it accessible for interaction.



In [10]:
print('Starting Ollama server in the background...')
!nohup ollama serve &
print('Ollama server started.')

Starting Ollama server in the background...
nohup: appending output to 'nohup.out'
Ollama server started.


**Reasoning**:
Now that the Ollama server has been started, I need to confirm that it is running and accessible. Listing the available models is a good way to verify the server's operational status and responsiveness.



In [11]:
print('Verifying Ollama server is running by listing models...')
!ollama list
print('Ollama server verification attempt complete.')

Verifying Ollama server is running by listing models...
NAME    ID    SIZE    MODIFIED 
Ollama server verification attempt complete.


**Reasoning**:
Since no models are currently available, I need to pull a model to be able to interact with Ollama using the `llama` function. I will pull the default 'llama2' model.



In [12]:
print('Pulling llama2 model...')
!ollama pull llama2
print('llama2 model pull complete.')

Pulling llama2 model...

llama2 model pull complete.


**Reasoning**:
Now that Ollama is installed, the server is running, and the 'llama2' model has been pulled, I need to verify that the `llama` function can successfully interact with it. I will re-run the initial prompt using the `llama` function to check for a successful response.



In [13]:
prompt = """
    What are fun activities I can do this weekend?
"""
response = llama(prompt)
print(response)


There are so many fun activities you can do this weekend, depending on your interests and preferences! Here are some ideas to get you started:

1. Outdoor Adventures:
	* Go hiking or camping in a nearby park or nature reserve.
	* Rent a kayak or paddleboard and explore a nearby lake or river.
	* Take a bike ride through a scenic trail or park.
2. Cultural Events:
	* Attend a concert or festival featuring your favorite music genre.
	* Visit a museum or art gallery to explore new works and learn something new.
	* Take a cooking class and learn how to make a new dish.
3. Sports and Fitness:
	* Join a pickup game of basketball or soccer with friends.
	* Take a yoga or fitness class to stay active and healthy.
	* Go for a run or a hike to enjoy the outdoors and get some exercise.
4. Indoor Activities:
	* Host a game night and play board games or card games with friends.
	* Have a movie marathon and watch your favorite films.
	* Try a new craft or hobby, like knitting or painting.
5. Spa Da